In [2]:
import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils
import pandas as pd

large_data_dir = gf_utils.large_data_dir

In [ ]:
all_df = pd.DataFrame()

for lib in ['1-plex','1','2','4-plex']: ### 1 and 2 are 2 parts of the 16-plex
    if lib == '4-plex':
        BCs = ['BC001', 'BC002', 'BC003', 'BC004']
    elif lib == '1-plex':
        BCs = ['BC001']
    else:
        BCs = ['BC001', 'BC002', 'BC003', 'BC004', 'BC005', 'BC006', 'BC007', 'BC008', 'BC009', 'BC010', 'BC011', 'BC012', 'BC013', 'BC014', 'BC015', 'BC016']

    for BC in BCs:
        sample = BC + '_' + lib

        pcr_swap_likelihoods = pd.read_csv('../data/swap_probabilities/likelihood_tables_' + lib.replace('-','') + '/patient_' + BC + '_swap_probabilities.csv')
        pcr_swap_likelihoods.rename(columns={'likelihood':'no_pcr_swap_likelihood'}, inplace=True)
        pcr_swap_likelihoods['pcr_swap_likelihood'] = 1 - pcr_swap_likelihoods['no_pcr_swap_likelihood']
        threshold = pcr_swap_likelihoods.loc[pcr_swap_likelihoods['pcr_swap_likelihood'] < 0.1]['pcr_duplicate_count'].min() - 1 ### set threshold 1 below so UMIs at threshold are included; same as high confidence counts elsewhere

        gf_dirs = {}
        ### first get probe_reads to use for the patient
        if lib == '4-plex':
            gf_dirs[0] = large_data_dir + 'gf_MPN/gf_MPN_4plex_1/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_4plex_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '1-plex':
            gf_dirs[0] = large_data_dir + 'gf_MPN/MPN_1plex/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '1': ## 16-plex
            gf_dirs[0] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_1/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'
        elif lib == '2': ## 16-plex
            gf_dirs[1] = large_data_dir + 'gf_MPN/gf_MPN_16plex_part' + lib + '_2/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'

        min_percent_supporting = 0.9
        collapse_across_probes = True
        iter = 0
        for key,gf_dir in gf_dirs.items():
            if iter == 0:
                probe_reads = gf_utils.get_input_probe_reads(gf_dir, read_threshold = threshold, cell_barcode_suffix = '-' + str(key), min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)
                manifest = gf_utils.get_manifest(gf_dir)
            else:
                probe_reads = pd.concat([probe_reads, gf_utils.get_input_probe_reads(gf_dir, read_threshold = threshold, cell_barcode_suffix = '-' + str(key), min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)], ignore_index=True)
            iter += 1       

        manifest['name'] = manifest['name'].str.replace('_1$', '', regex=True) ### fix name for dual probes which will end with _0 or _1
        manifest['name'] = manifest['name'].str.replace('_0$', '', regex=True) ### fix name for dual probes which will end with _0 or _1

        probe_reads = probe_reads.merge(manifest[['name']], left_on='probe_idx', right_index=True, how='left')

        df = pd.DataFrame({
            'n_UMIs': probe_reads.loc[probe_reads['name'].str.contains('c.')].groupby('barcode').size(),
            'n_sites_genotyped': probe_reads.loc[probe_reads['name'].str.contains('c.')].groupby('barcode')['probe_idx'].nunique(),
            'n_probes_in_panel': manifest.loc[manifest['name'].str.contains('c.')]['name'].nunique()
        })
        df.index = df.index + '-' + sample
        
        all_df = pd.concat([all_df, df])

12317 UMIs found
Collapsing UMIs across probes, 12317 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (6) and min percent supporting (0.9), 9724 UMIs remaining (78.95%)
568648 UMIs found
Collapsing UMIs across probes, 568648 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (18) and min percent supporting (0.9), 420431 UMIs remaining (73.94%)
674061 UMIs found
Collapsing UMIs across probes, 674061 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (18) and min percent supporting (0.9), 490991 UMIs remaining (72.84%)
387979 UMIs found
Collapsing UMIs across probes, 387979 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (17) and min percent supporting (0.9), 300059 UMIs remaining (77.34%)
447368 UMIs found
Collapsing UMIs across probes, 447368 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (17) and min percent supporting (0.9), 341718 UMIs remaining (76.38%)
616887 UMIs found
Co

In [10]:
all_df.to_csv('../output/MPN_UMIs_per_cell.csv')